In [1]:
import pandas as pd
import sqlalchemy as sal

In [5]:
engine = sal.create_engine('mssql+pyodbc://JUSTICES-NB\SQLEXPRESS/transactiondwh?driver=ODBC+DRIVER+17+FOR+SQL+SERVER')
conn = engine.connect()

In [56]:
# extract data from file and database 
def extract():
    df_customer = pd.read_csv('customer.txt')
    #convert datatypes 
    df_customer['sale_date'] = pd.to_datetime(df_customer['sale_date'])
    df_customer_db = pd.read_sql_query('select * from customers', con =conn)
    return df_customer_db, df_customer

def transformation(df_customer_db,df_customer):
    df_merge = pd.merge(df_customer, df_customer_db, how='left', on='cust_id')
# get results that are null or do not exist on the database 
    df_merge = df_merge[df_merge['product_id'].isna()]
    df_merge_final = df_merge.iloc[:,0:5]
    df_merge_final.columns = df_customer_db.columns
    
# update existing records where 
    df_updates = df_merge[df_merge['product_id'].notna()]
    df_updates = df_merge.iloc[:,0:5]
    df_updates.columns = df_customer_db.columns
    return df_merge_final,df_updates

# updated records will be inserted here 
def customer_staging_table(df_updates):
    df_merge_final.to_sql('customers_staging', con= conn, index=False, if_exists='replace')
    conn.commit()
    
# update existing records from staging to main table 
def updates():
    query = sal.text('''
     update customers 
     set cust_id = cs.cust_id, product_id = cs.product_id, sales_amount = cs.sales_amount, sale_date = cs.sale_date
     from customers_staging cs
     where customers.cust_id = cs.cust_id
    ''')
    conn.execute(query)
    conn.commit()

# insert new records from the table 
def insert(df_merge_final):
    df_merge_final.to_sql('customers', con= conn, index=False, if_exists='append')
    conn.commit()
    

In [57]:
# call all functions
df_customer_db,df_customer  = extract()
df_merge_final,df_updates = transformation(df_customer_db,df_customer)
customer_staging_table(df_updates)
updates()
insert(df_merge_final)